

# Directed Acyclic Graph

This example demonstrates how to create a random directed acyclic graph (DAG), which is useful in a number of contexts including for Git commit history.


In [ ]:
import igraph as ig
import matplotlib.pyplot as plt
import random

First, we set a random seed for reproducibility.



In [ ]:
random.seed(0)

First, we generate a random undirected graph with a fixed number of edges, without loops.



In [ ]:
g = ig.Graph.Erdos_Renyi(n=15, m=30, directed=False, loops=False)

Then we convert it to a DAG *in place*. This method samples DAGs with a given number of edges and vertices uniformly.



In [ ]:
g.to_directed(mode="acyclic")

We can print out a summary of the DAG.



In [ ]:
ig.summary(g)

Finally, we can plot the graph using the Sugiyama layout from :meth:`igraph.Graph.layout_sugiyama`:



In [ ]:
fig, ax = plt.subplots()
ig.plot(
    g,
    target=ax,
    layout="sugiyama",
    vertex_size=15,
    vertex_color="grey",
    edge_color="#222",
    edge_width=1,
)
plt.show()

In [12]:
import networkx as nx
import random


def generate_random_dag(num_nodes, edge_probability):
    G = nx.DiGraph()
    nodes = [f"Task_{i}" for i in range(num_nodes)]
    
    # Measured Historical Ranges
    # Values represent the spread across your two workflow families
    MEM_MIN, MEM_MAX = 3.26e6, 8.91e9
    TIME_MIN, TIME_MAX = 5.84e3, 2.84e8
    EDGE_MIN, EDGE_MAX = 7.36, 1.08e11

    # 'Swaps' distribution: 15% High, 45% Moderate, 40% Low
    swap_options = ['high', 'moderate', 'low']
    swap_weights = [0.15, 0.45, 0.40]

    # 1. Initialize Nodes with Telemetry-accurate values
    for i, node_name in enumerate(nodes):
        comp_cost = random.uniform(TIME_MIN, TIME_MAX)
        mem_cost = random.uniform(MEM_MIN, MEM_MAX)
        swap_val = random.choices(swap_options, weights=swap_weights)[0]
        
        G.add_node(
            node_name, 
            label=f"T{i}", 
            C=comp_cost, 
            M=mem_cost, 
            swaps=swap_val
        )

    # 2. Random Edge Generation
    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            if random.random() < edge_probability:
                weight = random.uniform(EDGE_MIN, EDGE_MAX)
                G.add_edge(nodes[i], nodes[j], weight=weight)

    # 3. Sequential Bridge (The "No Orphan" Rule)
    # Every node (except Task_0) must have at least one parent
    for j in range(1, num_nodes):
        if G.in_degree(nodes[j]) == 0:
            parent_idx = random.randint(0, j - 1)
            weight = random.uniform(EDGE_MIN, EDGE_MAX)
            G.add_edge(nodes[parent_idx], nodes[j], weight=weight)

    return G

In [13]:
import random
import os
# Configuration
sizes = [50, 100, 200, 1000, 2000, 4000, 8000, 10000, 15000, 20000, 25000, 30000]
samples_per_size = 5
output_dir = "random_dags"

os.makedirs(output_dir, exist_ok=True)

for size in sizes:
    print(f"--- Generating {samples_per_size} Random DAGs of size {size} ---")
    size_dir = os.path.join(output_dir, f"size_{size}")
    os.makedirs(size_dir, exist_ok=True)
    
    # Adjust probability based on size: 
    # Larger graphs need lower probability to avoid edge explosion.
    prob = max(0.001, 5.0 / size) 
    
    for i in range(samples_per_size):
        dag = generate_random_dag(size, edge_probability=prob)
        filename = os.path.join(size_dir, f"dag_{size}_{i}.dot")
        nx.drawing.nx_pydot.write_dot(dag, filename)
        
    print(f"Completed size {size}")

print(f"\nDone! Files are in '{output_dir}'.")

--- Generating 5 Random DAGs of size 50 ---
Completed size 50
--- Generating 5 Random DAGs of size 100 ---
Completed size 100
--- Generating 5 Random DAGs of size 200 ---
Completed size 200
--- Generating 5 Random DAGs of size 1000 ---
Completed size 1000
--- Generating 5 Random DAGs of size 2000 ---
Completed size 2000
--- Generating 5 Random DAGs of size 4000 ---
Completed size 4000
--- Generating 5 Random DAGs of size 8000 ---
Completed size 8000
--- Generating 5 Random DAGs of size 10000 ---
Completed size 10000
--- Generating 5 Random DAGs of size 15000 ---
Completed size 15000
--- Generating 5 Random DAGs of size 20000 ---
Completed size 20000
--- Generating 5 Random DAGs of size 25000 ---
Completed size 25000
--- Generating 5 Random DAGs of size 30000 ---
Completed size 30000

Done! Files are in 'random_dags'.


In [1]:
import networkx as nx
import random

def generate_shaped_dag(num_nodes, density=0.05, shape="general"):
    """
    shape options: 
    - "general": standard random DAG
    - "long": thin, deep dependencies (low parallelism)
    - "wide": short, fanned-out (high parallelism)
    """
    print("start generating")
    G = nx.DiGraph()
    nodes = [f"Task_{i}" for i in range(num_nodes)]
    
    # 1. Edge Generation based on Shape
    for j in range(1, num_nodes):
        # Determine the "Lookback Window"
        if shape == "long":
            # Only look back at the last 1-3 nodes
            start_lookback = max(0, j - 3)
        elif shape == "wide":
            # Prefer looking back at the first 10% of nodes
            start_lookback = 0
            j = max(1, int(num_nodes * 0.1)) 
        else:
            start_lookback = 0

        # Create edges based on density
        for i in range(start_lookback, j):
            if random.random() < density:
                G.add_edge(nodes[i], nodes[j])

    # 2. Sequential Bridge (Ensures Connectedness)
    for j in range(1, num_nodes):
        if G.in_degree(nodes[j]) == 0:
            if shape == "long":
                # Connect to the immediate predecessor for a tight chain
                parent_idx = j - 1
            elif shape == "wide":
                # Connect to one of the initial root tasks
                parent_idx = random.randint(0, max(0, int(num_nodes * 0.05)))
            else:
                parent_idx = random.randint(0, j - 1)
                
            G.add_edge(nodes[parent_idx], nodes[j])

    return G

In [3]:
import networkx as nx
import os
print("start")

# --- Configuration ---
sizes = [50, 200, 1000, 2000, 4000, 8000, 10000, 15000, 20000, 25000, 30000]#
families = [ "general" ]#"general","long", "wide"
iterations = 5  # 5 pieces per size per family
output_dir = "./generated_dags"

print("start")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# --- Execution Loop ---
for size in sizes:
    print("for")
    for family in families:
        print("for")
        for i in range(iterations):
            print("for")
            # 1. Adjust density based on family
            # 'Long' needs very low density to stay thin
            # 'General' can handle more complexity
            if family == "long":
                p = 0.02 
            elif family == "wide":
                p = 0.08
            else:
                p = 0.05
            
            # 2. Call the generator
            # (Assuming generate_shaped_dag is defined in your environment)
            G = generate_shaped_dag(size, density=p, shape=family)
            
            # 3. Define Filename
            # Format: dag_SIZE_FAMILY_INDEX.dot
            filename = f"dag_lowio_{size}_{family}_{i}.dot"
            filepath = os.path.join(output_dir, filename)
            
            # 4. Export to DOT
            # Note: requires 'pydot' or 'pygraphviz' installed
            try:
                nx.drawing.nx_pydot.write_dot(G, filepath)
                print(f"Successfully generated: {filename}")
            except ImportError:
                # Fallback if pydot isn't installed
                nx.write_network_text(G) 
                print("Error: Install pydot to save .dot files.")

print("\nGeneration Complete!")

start
start
for
for
for
start generating
Successfully generated: dag_25000_long_0.dot
for
start generating
Successfully generated: dag_25000_long_1.dot
for
start generating
Successfully generated: dag_25000_long_2.dot
for
start generating
Successfully generated: dag_25000_long_3.dot
for
start generating
Successfully generated: dag_25000_long_4.dot
for
for
start generating
Successfully generated: dag_25000_wide_0.dot
for
start generating
Successfully generated: dag_25000_wide_1.dot
for
start generating
Successfully generated: dag_25000_wide_2.dot
for
start generating
Successfully generated: dag_25000_wide_3.dot
for
start generating
Successfully generated: dag_25000_wide_4.dot
for
for
for
start generating
Successfully generated: dag_30000_long_0.dot
for
start generating
Successfully generated: dag_30000_long_1.dot
for
start generating
Successfully generated: dag_30000_long_2.dot
for
start generating
Successfully generated: dag_30000_long_3.dot
for
start generating
Successfully generated:

In [4]:
import networkx as nx
import random

def generate_shaped_dag(num_nodes, edge_probability, shape="general"):
    G = nx.DiGraph()
    nodes = [f"Task_{i}" for i in range(num_nodes)]
    
    # Measured Historical Ranges
    MEM_MIN, MEM_MAX = 3.26e6, 8.91e9
    TIME_MIN, TIME_MAX = 5.84e3, 2.84e8
    EDGE_MIN, EDGE_MAX = 7.36, 1.08e4

    # 'Swaps' distribution: 15% High, 45% Moderate, 40% Low
    swap_options = ['high', 'moderate', 'low']
    swap_weights = [0.15, 0.45, 0.40]

    # 1. Initialize Nodes
    for i, node_name in enumerate(nodes):
        comp_cost = random.uniform(TIME_MIN, TIME_MAX)
        mem_cost = random.uniform(MEM_MIN, MEM_MAX)
        swap_val = random.choices(swap_options, weights=swap_weights)[0]
        
        G.add_node(
            node_name, 
            label=f"T{i}", 
            C=comp_cost, 
            M=mem_cost, 
            swaps=swap_val
        )

    # 2. Shaped Edge Generation
    for j in range(1, num_nodes):
        # Define the lookback window based on shape
        if shape == "long":
            # Forces a deep, thin chain by only looking back 1-3 steps
            start_lookback = max(0, j - 3)
        elif shape == "wide":
            # Forces a shallow, wide fan-out by preferring early 'root' nodes
            # We restrict the potential parents to the first 10% of the graph
            start_lookback = 0
            # For wide, we don't want to look at EVERYTHING, just the 'roots'
            j_limit = max(1, int(num_nodes * 0.1)) 
            if j > j_limit:
                # If we are past the root phase, only allow connecting to roots
                search_range = range(0, j_limit)
            else:
                search_range = range(0, j)
        else:
            # "General" - can connect to any previous node
            search_range = range(0, j)

        if shape != "wide":
            search_range = range(start_lookback, j)

        for i in search_range:
            if random.random() < edge_probability:
                weight = random.uniform(EDGE_MIN, EDGE_MAX)
                G.add_edge(nodes[i], nodes[j], weight=weight)

    # 3. Sequential Bridge (The "No Orphan" Rule)
    for j in range(1, num_nodes):
        if G.in_degree(nodes[j]) == 0:
            if shape == "long":
                parent_idx = j - 1 # Tightest possible chain
            elif shape == "wide":
                # Connect orphans back to the very first few nodes
                parent_idx = random.randint(0, max(0, int(num_nodes * 0.05)))
            else:
                parent_idx = random.randint(0, j - 1)
            
            weight = random.uniform(EDGE_MIN, EDGE_MAX)
            G.add_edge(nodes[parent_idx], nodes[j], weight=weight)

    return G

In [5]:
import os

# --- Setup ---
sizes = [ 50, 100, 200, 1000, 2000, 4000, 8000, 10000, 15000, 20000, 25000, 30000]#
families = ["general" ]# "general", "long", "wide"
iterations = 2
output_folder = "dag_outputs"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# --- Generation Loop ---
for size in sizes:
    for fam in families:
        print("start")
        # Tweak probability so families don't accidentally look the same
        # Long/Thin needs lower prob to prevent "ribbon" widening
        # Wide needs decent prob to ensure it actually fans out
        prob = 0.02 if fam == "long" else (0.1 if fam == "wide" else 0.05)
        
        for i in range(iterations):
            G = generate_shaped_dag(size, prob, shape=fam)
            
            # Save the file
            filename = f"dag_lowio_{size}_{fam}_{i}.dot"
            path = os.path.join(output_folder, filename)
            
            # Write to DOT with pydot
            nx.drawing.nx_pydot.write_dot(G, path)
            print(f"Generated {path}")

print("\nAll tasks complete. Weights, C, M, and Swaps are all present.")

start


UnboundLocalError: local variable 'start_lookback' referenced before assignment

In [1]:
import networkx as nx
import random

def generate_shaped_dag(num_nodes, edge_probability, shape="general"):
    G = nx.DiGraph()
    nodes = [f"Task_{i}" for i in range(num_nodes)]
    
    # Historical Ranges
   #MEM_MIN, MEM_MAX = 3.26e7, 8.91e11#  8.91e9for normal
   # TIME_MIN, TIME_MAX = 5.84e3, 2.84e8
   # EDGE_MIN, EDGE_MAX = 3.26e2, 1.08e5#change to 5 for low io, change lower to 3.26e6 for high edge

    MEM_MIN, MEM_MAX = 3.26e6, 8.91e9
    TIME_MIN, TIME_MAX = 5.84e3, 2.84e8
    EDGE_MIN, EDGE_MAX = 7.36, 1.08e11

    swap_options = ['high', 'moderate', 'low']
    swap_weights = [0.15, 0.45, 0.40]

    for i, node_name in enumerate(nodes):
        G.add_node(node_name, label=f"T{i}", 
                   C=random.uniform(TIME_MIN, TIME_MAX), 
                   M=random.uniform(MEM_MIN, MEM_MAX), 
                   swaps=random.choices(swap_options, weights=swap_weights)[0])

    # 1. Structured Edge Generation
    for j in range(1, num_nodes):
        # STRICT RULE: Potential parents MUST have index i < j
        if shape == "long":
            # Only look back at the immediate 3 predecessors
            parent_pool = range(max(0, j - 3), j)
        elif shape == "wide":
            # Prefer the first 10% of nodes as roots for everyone
            root_limit = max(1, int(num_nodes * 0.1))
            if j <= root_limit:
                parent_pool = range(0, j)
            else:
                parent_pool = range(0, root_limit)
        else:
            parent_pool = range(0, j)

        for i in parent_pool:
            if random.random() < edge_probability:
                # Double check i < j to be absolutely safe
                if i < j:
                    weight = random.uniform(EDGE_MIN, EDGE_MAX)
                    G.add_edge(nodes[i], nodes[j], weight=weight)

    # 2. Sequential Bridge (Ensures Connectedness & Zero Cycles)
    for j in range(1, num_nodes):
        if G.in_degree(nodes[j]) == 0:
            if shape == "long":
                parent_idx = j - 1
            elif shape == "wide":
                # Connect to a random node in the root prefix
                root_limit = max(1, int(num_nodes * 0.1))
                # Ensure the parent_idx is ALWAYS less than the current node j
                upper_bound = min(j - 1, root_limit - 1)
                parent_idx = random.randint(0, max(0, upper_bound))
            else:
                parent_idx = random.randint(0, j - 1)
            
            weight = random.uniform(EDGE_MIN, EDGE_MAX)
            G.add_edge(nodes[parent_idx], nodes[j], weight=weight)

    return G

import os

# --- Setup ---
sizes = [ 50,  200, 1000, 2000, 4000, 8000, 10000, 15000, 20000]#
families = ["general", "long", "wide" ]# "general", "long", "wide"
iterations = 5
output_folder = "dag_outputs_3"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# --- Generation Loop ---
for size in sizes:
    for fam in families:
        print("start")
        # Tweak probability so families don't accidentally look the same
        # Long/Thin needs lower prob to prevent "ribbon" widening
        # Wide needs decent prob to ensure it actually fans out
        prob = 0.02 # if fam == "long" else (0.05 if fam == "wide" else 0.05) # used to be 0.1 for wide
        
        for i in range(iterations):
            G = generate_shaped_dag(size, prob, shape=fam)
            
            # Save the file
            filename = f"dag_{size}_{fam}_{i}.dot"
            path = os.path.join(output_folder, filename)
            
            # Write to DOT with pydot
            nx.drawing.nx_pydot.write_dot(G, path)
            print(f"Generated {path}")

print("\nAll tasks complete. Weights, C, M, and Swaps are all present.")


start
Generated dag_outputs_3/dag_50_general_0.dot
Generated dag_outputs_3/dag_50_general_1.dot
Generated dag_outputs_3/dag_50_general_2.dot
Generated dag_outputs_3/dag_50_general_3.dot
Generated dag_outputs_3/dag_50_general_4.dot
start
Generated dag_outputs_3/dag_50_long_0.dot
Generated dag_outputs_3/dag_50_long_1.dot
Generated dag_outputs_3/dag_50_long_2.dot
Generated dag_outputs_3/dag_50_long_3.dot
Generated dag_outputs_3/dag_50_long_4.dot
start
Generated dag_outputs_3/dag_50_wide_0.dot
Generated dag_outputs_3/dag_50_wide_1.dot
Generated dag_outputs_3/dag_50_wide_2.dot
Generated dag_outputs_3/dag_50_wide_3.dot
Generated dag_outputs_3/dag_50_wide_4.dot
start
Generated dag_outputs_3/dag_200_general_0.dot
Generated dag_outputs_3/dag_200_general_1.dot
Generated dag_outputs_3/dag_200_general_2.dot
Generated dag_outputs_3/dag_200_general_3.dot
Generated dag_outputs_3/dag_200_general_4.dot
start
Generated dag_outputs_3/dag_200_long_0.dot
Generated dag_outputs_3/dag_200_long_1.dot
Generated

In [4]:
import networkx as nx
import random

def generate_shaped_dag(num_nodes, edge_probability, shape="general"):
    G = nx.DiGraph()
    nodes = [f"Task_{i}" for i in range(num_nodes)]
    
    # Historical Ranges
   #MEM_MIN, MEM_MAX = 3.26e7, 8.91e11#  8.91e9for normal
   # TIME_MIN, TIME_MAX = 5.84e3, 2.84e8
   # EDGE_MIN, EDGE_MAX = 3.26e2, 1.08e5#change to 5 for low io, change lower to 3.26e6 for high edge

    MEM_MIN, MEM_MAX = 3.26e6, 8.91e9
    TIME_MIN, TIME_MAX = 5.84e3, 2.84e8
    EDGE_MIN, EDGE_MAX = 7.36, 1.08e11

    swap_options = ['high', 'moderate', 'low']
    swap_weights = [0.15, 0.45, 0.40]

    for i, node_name in enumerate(nodes):
        G.add_node(node_name, label=f"T{i}", 
                   C=random.uniform(TIME_MIN, TIME_MAX), 
                   M=random.uniform(MEM_MIN, MEM_MAX), 
                   swaps=random.choices(swap_options, weights=swap_weights)[0])

    # 1. Structured Edge Generation
    for j in range(1, num_nodes):
        # STRICT RULE: Potential parents MUST have index i < j
        if shape == "long":
            # Only look back at the immediate 3 predecessors
            parent_pool = range(max(0, j - 3), j)
        elif shape == "wide":
            # Prefer the first 10% of nodes as roots for everyone
            root_limit = max(1, int(num_nodes * 0.1))
            if j <= root_limit:
                parent_pool = range(0, j)
            else:
                parent_pool = range(0, root_limit)
        else:
            parent_pool = range(0, j)

        for i in parent_pool:
            if random.random() < edge_probability:
                # Double check i < j to be absolutely safe
                if i < j:
                    weight = random.uniform(EDGE_MIN, EDGE_MAX)
                    G.add_edge(nodes[i], nodes[j], weight=weight)

    # 2. Sequential Bridge (Ensures Connectedness & Zero Cycles)
    for j in range(1, num_nodes):
        if G.in_degree(nodes[j]) == 0:
            if shape == "long":
                parent_idx = j - 1
            elif shape == "wide":
                # Connect to a random node in the root prefix
                root_limit = max(1, int(num_nodes * 0.1))
                # Ensure the parent_idx is ALWAYS less than the current node j
                upper_bound = min(j - 1, root_limit - 1)
                parent_idx = random.randint(0, max(0, upper_bound))
            else:
                parent_idx = random.randint(0, j - 1)
            
            weight = random.uniform(EDGE_MIN, EDGE_MAX)
            G.add_edge(nodes[parent_idx], nodes[j], weight=weight)

    return G

import os

# --- Setup ---
sizes = [ 50,  200, 1000, 2000, 4000, 8000, 10000, 15000, 20000, 25000, 30000 ]#
families = ["general", "long", "wide" ]# "general", "long", "wide"
iterations = 5
output_folder = "dag_outputs_4"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)


# --- Generation Loop ---
for size in sizes:
    # Target approximately 1.8 edges per node to match your 20K -> 36K example
    # We use (size * 1.8) / (size * (size - 1) / 2) which simplifies to 3.6 / size
    base_prob = 4 / size 

    for fam in families:
        print(f"Generating {size} - {fam}")
        
        # Adjust probability slightly by family to maintain their "shapes"
        if fam == "long":
            # Long dags are naturally sparse; keep it very low
            prob = 2 / size 
        elif fam == "wide":
            # Wide dags need a few more connections to "fan out"
            prob = 4.0 / size
        else:
            prob = base_prob
        
        # Ensure probability never exceeds 1.0 (for very small sizes)
        prob = min(prob, 0.5)

        for i in range(iterations):
            G = generate_shaped_dag(size, prob, shape=fam)
             # Save the file
            filename = f"dag_{size}_{fam}_{i}.dot"
            path = os.path.join(output_folder, filename)
            
            # Write to DOT with pydot
            nx.drawing.nx_pydot.write_dot(G, path)
            print(f"Generated {path}")           
            
            
print("\nAll tasks complete. Weights, C, M, and Swaps are all present.")


Generating 50 - general
Generated dag_outputs_4/dag_50_general_0.dot
Generated dag_outputs_4/dag_50_general_1.dot
Generated dag_outputs_4/dag_50_general_2.dot
Generated dag_outputs_4/dag_50_general_3.dot
Generated dag_outputs_4/dag_50_general_4.dot
Generating 50 - long
Generated dag_outputs_4/dag_50_long_0.dot
Generated dag_outputs_4/dag_50_long_1.dot
Generated dag_outputs_4/dag_50_long_2.dot
Generated dag_outputs_4/dag_50_long_3.dot
Generated dag_outputs_4/dag_50_long_4.dot
Generating 50 - wide
Generated dag_outputs_4/dag_50_wide_0.dot
Generated dag_outputs_4/dag_50_wide_1.dot
Generated dag_outputs_4/dag_50_wide_2.dot
Generated dag_outputs_4/dag_50_wide_3.dot
Generated dag_outputs_4/dag_50_wide_4.dot
Generating 200 - general
Generated dag_outputs_4/dag_200_general_0.dot
Generated dag_outputs_4/dag_200_general_1.dot
Generated dag_outputs_4/dag_200_general_2.dot
Generated dag_outputs_4/dag_200_general_3.dot
Generated dag_outputs_4/dag_200_general_4.dot
Generating 200 - long
Generated da